# Baseline CVAE Quick Test

Run the cells from top to bottom for a very simple sanity check.
This trains for 1 epoch, then generates a few peptides.

In [10]:
from pathlib import Path
import pickle

import torch

from training import train_model
from model import PeptideCVAE
from inference import generate_peptides

In [21]:
# Paths relative to this notebook
baseline_dir = Path(".")
dataset_file = "../database/test.json"
model_path = baseline_dir / "cvae_peptide_model.pth"
scaler_path = baseline_dir / "scaler.pkl"

# Quick 1-epoch train
model, device = train_model(
    dataset_file=str(dataset_file),
    epochs=70,
    batch_size=32,
    max_len=12,
    latent_dim=32,
    model_path=str(model_path),
)

print(f"Model saved at: {model_path}")
print(f"Scaler expected at: {scaler_path}")

Using device: cuda
Scaler saved to scaler.pkl
Epoch [1/70] | Beta: 0.00 | Loss: 21.6298 | Recon: 21.6298 | KL: 57.9305
Epoch [5/70] | Beta: 0.11 | Loss: 13.7313 | Recon: 11.8930 | KL: 16.0848
Epoch [10/70] | Beta: 0.26 | Loss: 12.0176 | Recon: 9.2979 | KL: 10.5766
Epoch [15/70] | Beta: 0.40 | Loss: 11.5097 | Recon: 8.3413 | KL: 7.9212
Epoch [20/70] | Beta: 0.54 | Loss: 11.3181 | Recon: 8.0249 | KL: 6.0664
Epoch [25/70] | Beta: 0.69 | Loss: 11.1509 | Recon: 7.9566 | KL: 4.6583
Epoch [30/70] | Beta: 0.83 | Loss: 10.7516 | Recon: 8.0105 | KL: 3.3082
Epoch [35/70] | Beta: 0.97 | Loss: 10.3504 | Recon: 8.0962 | KL: 2.3205
Epoch [40/70] | Beta: 1.00 | Loss: 9.5244 | Recon: 7.8362 | KL: 1.6882
Epoch [45/70] | Beta: 1.00 | Loss: 8.7421 | Recon: 7.3884 | KL: 1.3537
Epoch [50/70] | Beta: 1.00 | Loss: 7.9509 | Recon: 6.8466 | KL: 1.1043
Epoch [55/70] | Beta: 1.00 | Loss: 7.2922 | Recon: 6.3776 | KL: 0.9146
Epoch [60/70] | Beta: 1.00 | Loss: 6.7790 | Recon: 5.9729 | KL: 0.8061
Epoch [65/70] | Beta

In [ ]:
# Reload model + scaler and generate a few peptides
with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)

# Keep architecture args aligned with training defaults
loaded_model = PeptideCVAE(
    vocab_size=24,
    condition_dim=8,
    max_seq_len=14,
    latent_dim=32,
)
loaded_model.load_state_dict(torch.load(model_path, map_location=device))

target = [11, 7, 1567.99, -0.31, 6.0, 14.0, 0.51, 5]

samples = generate_peptides(
    model=loaded_model,
    scaler=scaler,
    num_samples=5,
    properties=target,
    temperature=0.9,
    top_k=5,
)

samples

Using device: cuda
Model loaded successfully.

Target Properties:
- Length: 11.0
- pH: 7.0
- Molecular Weight: 1567.99
- LogP: -0.31
- Net Charge: 6.0
- Isoelectric Point: 14.0
- Hydrophobicity: 0.51
- Cathionicity: 5.0

--- Generating 5 novel peptide(s) ---
Sample 1: KKLLRWIRWLR
Sample 2: LRFLKKWRFKLK
Sample 3: KWLKRIKTLLK
Sample 4: KWLKRIKTLLK
Sample 5: KWLKKVVKWLR


['KKLLRWIRWLR', 'LRFLKKWRFKLK', 'KWLKRIKTLLK', 'KWLKRIKTLLK', 'KWLKKVVKWLR']